# 🏎️ Notebook 10 — The Steering Wheel
### Continuous Actions with PPO

**Series**: RL Notebook Series · Act IV — Engineering · Post 10 of 16

---

## What happens when we can't just press buttons?

Up until this very moment, every single algorithm we've built (
Q-Learning, DQN, REINFORCE, Actor-Critic, PPO) has controlled the environment using **Discrete Actions**.
The environment gave us a list of generic buttons (0, 1, 2, 3), and our Neural Network used a `Softmax` layer and a `Categorical` distribution to output a probability for each button.

But what if we are trying to train a real physical Robot? Or a self driving car?

You can't just press the "Left" button. You have to turn the steering wheel exactly `-12.5` degrees. You have to apply exactly `14.2` Newtons of torque to the right knee joint.

These are **Continuous Actions**. We can't use a `Categorical` distribution anymore because there are infinitely many decimals between `-12.5` and `-12.6`! 

## The Gaussian Policy

Instead of outputting discrete probabilities, our Actor network will instead output the parameters of a **Bell Curve** (a Normal/Gaussian Distribution) for every single actuator on our robot.

For every continuous action, our Neural Network will output **two numbers**:
1. **The Mean ($\mu$)**: What exactly does the network think the action should be? (e.g., Turn steering wheel 12.5 degrees).
2. **The Standard Deviation ($\sigma$)**: How confident is the network in that action? Large $\sigma$ = Highly uncertain (explore randomly). Small $\sigma$ = Highly confident (act precisely).

We then build a `torch.distributions.Normal(mu, std)` distribution and sample an action from that Bell Curve!

### Let's drive!
We will be training an agent to swing up and balance an inverted pendulum using `Pendulum-v1`. The action is a single continuous float representing motor torque between `[-2.0, 2.0]`.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.grid': True, 'grid.alpha': 0.3})

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. The Continuous Actor Network

Notice the three major differences from Notebook 09:
1. The `Actor` now has a `mu_head` and a `log_std` parameter instead of outputting generic logits.
2. We pass the network's `mu` output through a `tanh()` activation. `tanh` bounds the output perfectly between `[-1, 1]`. We then multiply it by the environment's `max_action` (e.g. `2.0` for Pendulum) so the network inherently knows the physical safety limits of the motor!
3. We use `log_std` instead of `std` to ensure the standard deviation is strictly positive when we do `torch.exp(log_std)`.

In [3]:
class ContinuousPPOActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim, max_action, hidden_dim=256):
        super(ContinuousPPOActorCritic, self).__init__()
        self.max_action = max_action
        
        # Shared Features
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        
        # Critic Head (Outputs a single V(s) value, same as always!)
        self.critic_fc = nn.Linear(hidden_dim, 1)
        
        # The Continuous Actor Heads
        # 1. Output the Mean over the Action Space
        self.mu_head = nn.Linear(hidden_dim, action_dim)
        
        # 2. To Output the Standard Deviation (Uncertainty), we usually use a freestanding parameter
        # that isn't connected to the state directly. It learns globally how much to explore.
        # We start it at 0, which means torch.exp(0) = 1.0 standard deviation (plenty of early exploration).
        self.log_std = nn.Parameter(torch.zeros(1, action_dim))
        
    def forward(self, state):
        x = F.relu(self.fc1(state))
        
        # Value (Critic)
        state_value = self.critic_fc(x)
        
        # TODO: Calculate the Action Mean (mu)
        # 1. Pass the features (x) through the mu_head.
        # 2. Use torch.tanh to bound the output between [-1, 1].
        # 3. Multiply by self.max_action to scale it to the physical limits of the robot.
        mu = None
        
        # TODO: Calculate the Standard Deviation (std)
        # 1. We have a naked parameter self.log_std.
        # 2. Use torch.exp() to ensure the standard deviation is strictly positive.
        # 3. Expand the std to match the shape of mu uing `.expand_as(mu)`.
        std = None
        
        raise NotImplementedError("Complete the continuous forward pass!")
        
        return mu, std, state_value

    def evaluate(self, state, action):
        """Used during the PPO update loop to get log_probs and entropy of past actions"""
        mu, std, state_value = self.forward(state)
        
        # TODO: Create the continuous Gaussian Bell Curve!
        # Hint: Use the `Normal(mean, std)` class imported at the top of the file.
        dist = None
        
        # TODO: Calculate the log probability of the exact continuous action we took
        # Since continuous dimensions are independent, we `.sum(dim=-1)` across the action dimension
        # to get the total log probability of the joint action vector.
        log_prob = None
        
        # TODO: Calculate the Entropy (How "wide" or uncertain the bell curve is)
        # Don't forget to `.sum(dim=-1)` here too!
        entropy = None
        
        return log_prob, state_value, entropy

## 2. Re-using our trusted old PPO code!

Except for substituting `m = Normal(mu, std)` instead of `m = Categorical(probs)`, the **entire remaining PPO algorithm is exactly identical to Notebook 09**. 

The Advantage math, the clipped surrogate objective, the learning rates... none of it changes! The math works exactly the same whether it is a discrete Categorical log probability or a continuous Gaussian log probability.

In [4]:
class RolloutBuffer:
    def __init__(self):
        self.clear()
        
    def clear(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
        self.dones = []

In [5]:
def train_continuous_ppo(env_name='Pendulum-v1', num_episodes=500, gamma=0.99, lr=1e-3, K_epochs=10, eps_clip=0.2):
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]  # Note this is now .shape[0] instead of .n
    max_action = float(env.action_space.high[0]) # We read the max torque limit directly from the environment
    
    # Note: We use a smaller learning rate since Continuous spaces are extremely delicate
    network = ContinuousPPOActorCritic(state_dim, action_dim, max_action)
    optimizer = optim.Adam(network.parameters(), lr=lr) 
    memory = RolloutBuffer()
    
    episode_rewards = []
    update_interval = 10 # Update every 10 episodes (2000 steps)
    ep_count = 0
    
    for ep in range(num_episodes):
        state, _ = env.reset()
        done = False
        truncated = False
        ep_reward = 0
        
        while not (done or truncated):
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                mu, std, value = network(state_tensor)
            
            # Setup the continuous Distribution
            dist = Normal(mu, std)
            
            # Sample a continuous float! (e.g. 1.3412351)
            action = dist.sample()
            
            # We clamp the action to make sure we didn't mathematically sample an
            # action outside the robot's physical torque limits.
            action_clamped = torch.clamp(action, -max_action, max_action)
            
            log_prob = dist.log_prob(action).sum(dim=-1)
            
            # Environment expects a numpy array!
            next_state, reward, done, truncated, _ = env.step(action_clamped.numpy()[0])
            
            memory.states.append(state)
            memory.actions.append(action[0].numpy())
            memory.log_probs.append(log_prob.item())
            memory.values.append(value.item())
            memory.rewards.append((reward + 8.0) / 8.0) # Reward scaling for Pendulum
            memory.dones.append(done or truncated)  
            
            state = next_state
            ep_reward += reward
            
        episode_rewards.append(ep_reward)
        ep_count += 1
        
        # PPO Update Block 
        # (EXACTLY THE SAME AS NOTEBOOK 09!!!)
        if ep_count % update_interval == 0:
            old_states = torch.FloatTensor(np.array(memory.states))
            old_actions = torch.FloatTensor(np.array(memory.actions))
            old_log_probs = torch.FloatTensor(np.array(memory.log_probs))
            old_values = torch.FloatTensor(np.array(memory.values)).squeeze()
            
            returns = []
            discounted_reward = 0
            for reward, is_terminal in zip(reversed(memory.rewards), reversed(memory.dones)):
                if is_terminal:
                    discounted_reward = 0
                discounted_reward = reward + (gamma * discounted_reward)
                returns.insert(0, discounted_reward)
                
            returns = torch.FloatTensor(returns)
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
            
            advantages = returns - old_values
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
            
            for _ in range(K_epochs):
                new_log_probs, new_values, entropy = network.evaluate(old_states, old_actions)
                new_values = new_values.squeeze()
                
                ratios = torch.exp(new_log_probs - old_log_probs.detach())
                
                surr1 = ratios * advantages.detach()
                surr2 = torch.clamp(ratios, 1 - eps_clip, 1 + eps_clip) * advantages.detach()
                
                actor_loss = -torch.min(surr1, surr2).mean()
                critic_loss = 0.5 * F.mse_loss(new_values, returns.detach())
                entropy_loss = -0.01 * entropy.mean()
                
                loss = actor_loss + critic_loss + entropy_loss
                
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(network.parameters(), 0.5) 
                optimizer.step()
                
            memory.clear()
            
        if (ep + 1) % 50 == 0:
            avg = np.mean(episode_rewards[-50:])
            print(f"Episode {ep+1:3d} | Avg Reward: {avg:.1f}")
            if avg > -200:
                print("Solved!")
                break
            
    return network, episode_rewards

## 3. Training the Pendulum

In `Pendulum-v1`, the maximum possible reward is `0.0` (perfect balance at top-dead-center forever). The score represents a negative cost (how much energy it used and how far it deviated). An average reward above `-200` represents an excellent, optimally balancing agent.

**Note:** Pendulum is a notoriously difficult environment to perfect without massive amounts of data. In 500 episodes, you'll see the agent learn to swing the pendulum up (the score will improve from -1500 to around -1000), but it will likely struggle to balance it perfectly at the top. This is the exact reason we need the Distributed RL architectures coming in Notebook 11!

In [ ]:
trained_net, cp_rewards = train_continuous_ppo(num_episodes=500)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cp_rewards, color='#3b82f6', alpha=0.3)

window = 20
if len(cp_rewards) >= window:
    smoothed = np.convolve(cp_rewards, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, len(cp_rewards)), smoothed, color='#3b82f6', linewidth=2, label=f'Continuous PPO (Moving Avg)')

ax.set_title("Continuous PPO Training on Pendulum-v1")
ax.set_xlabel("Episode")
ax.set_ylabel("Total Reward")
ax.axhline(0, color='gray', linestyle='--', label="Theoretical Maximum")
ax.axhline(-200, color='green', linestyle='--', label="Solved Threshold")
ax.legend()
plt.show()

NameError: name 'train_continuous_ppo' is not defined

## 4. Let's Watch the Physics in Action

We use the trained agent's `mu` output exactly (no random sampling required at test time!) to control the exact torque.

In [7]:
def render_continuous_agent(env_name, agent):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset(seed=42)
    frames = []
    
    done = False
    truncated = False
    while not (done or truncated) and len(frames) < 300:
        frames.append(env.render())
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            # At test time, we ACT GREEDILY. 
            # We ignore the random sampler and standard deviation entirely. 
            # We just take the pure, unadulterated MU (Mean) output!
            mu, _, _ = agent(state_tensor)
            action = mu.numpy()[0]
            action_clamped = np.clip(action, env.action_space.low[0], env.action_space.high[0])
        state, _, done, truncated, _ = env.step(action_clamped)
    env.close()
    
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.axis('off')
    img = ax.imshow(frames[0])
    
    def animate(i):
        img.set_data(frames[i])
        return [img]
        
    anim = animation.FuncAnimation(fig, animate, frames=len(frames), interval=50, blit=True)
    plt.close() 
    return HTML(anim.to_jshtml())

render_continuous_agent('Pendulum-v1', trained_net)


## Conclusion of Act IV (Part 1)

You have successfully bridged the gap from playing Atari games with discrete buttons to controlling physics-based torque loops for continuous physical control.

What's remarkable is that **the foundational RL equations do not change.** Whether you are throwing dice (Categorical) or painting a Bell Curve (Gaussian), PPO operates perfectly either way.

However, as you saw, the agent learned *slowly*. Continuous Control is mathematically complex and requires a massive amount of exploration data to perfect. A single agent struggling for 500 episodes isn't quite enough to become a master.

What do we do when we want to train these state-of-the-art algorithms on giant clusters of computers at the same time to gather 100x more data per second? Let's dive into **Distributed RL**!